# Laboratório — Métricas de classificação

Neste laboratório, rótulos sintéticos tornam cada célula da matriz de confusão auditável. O foco é avaliar uma política de decisão já fixada; curvas, escolha de threshold e calibração ficam para a Aula 15.

**Objetivos:** reconstruir métricas, demonstrar o efeito da prevalência, comparar agregações multiclasse e auditar grupos.

Seed fixa: `20260908`. Nenhum dado externo ou credencial é usado.

## Ambiente

Dependências mínimas: Python 3.10, NumPy 1.24, pandas 2.0, Matplotlib 3.7 e scikit-learn 1.3.

No Colab, se necessário: `pip install "numpy>=1.24" "pandas>=2.0" "matplotlib>=3.7" "scikit-learn>=1.3"`.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)

SEED = 20260908
rng = np.random.default_rng(SEED)

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

## 1. Conjunto binário controlado

Construiremos exatamente `TP=40`, `FP=10`, `FN=20` e `TN=930`. Em seguida, embaralharemos os pares sem alterar as contagens.

In [ ]:
y_true = np.r_[np.ones(40), np.zeros(10), np.ones(20), np.zeros(930)].astype(int)
y_pred = np.r_[np.ones(40), np.ones(10), np.zeros(20), np.zeros(930)].astype(int)

ordem = rng.permutation(len(y_true))
y_true = y_true[ordem]
y_pred = y_pred[ordem]

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
print({"TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)})
assert (tp, fp, fn, tn) == (40, 10, 20, 930)

## 2. Fórmulas manuais e biblioteca

A classe positiva é `1`. Recall e specificity condicionam pela classe real; precision e NPV condicionam pela decisão.

In [ ]:
def metricas_binarias(tn, fp, fn, tp):
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    specificity = tn / (tn + fp)
    npv = tn / (tn + fn)
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    balanced = (recall + specificity) / 2
    f1 = 2 * tp / (2 * tp + fp + fn)
    return {
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "NPV": npv,
        "accuracy": accuracy,
        "balanced_accuracy": balanced,
        "F1": f1,
    }


manual = metricas_binarias(tn, fp, fn, tp)
biblioteca = {
    "precision": precision_score(y_true, y_pred),
    "recall": recall_score(y_true, y_pred),
    "specificity": recall_score(y_true, y_pred, pos_label=0),
    "NPV": precision_score(y_true, y_pred, pos_label=0),
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "F1": f1_score(y_true, y_pred),
}

for nome in manual:
    np.testing.assert_allclose(manual[nome], biblioteca[nome], atol=1e-12)
pd.DataFrame({"manual": manual, "scikit-learn": biblioteca}).round(6)

**Resultados esperados:** precision `0.8`, recall `0.666667`, specificity `0.989362`, NPV `0.978947`, accuracy `0.97`, balanced accuracy `0.828014` e F1 `0.727273`.

## 3. Matriz com orientação explícita

**Texto alternativo do gráfico:** matriz 2×2; linha real 0 contém 930 previsões 0 e 10 previsões 1; linha real 1 contém 20 previsões 0 e 40 previsões 1.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, labels=[0, 1], display_labels=["negativo", "positivo"],
    cmap="Blues", colorbar=False, ax=ax,
)
ax.set_title("Linhas reais, colunas previstas")
plt.tight_layout()
plt.show()

## 4. Baseline sempre negativo

Com 60 positivos em mil casos, a regra sempre negativa obtém accuracy alta, mas não detecta nenhum positivo. Definimos `zero_division=0` deliberadamente.

In [ ]:
pred_baseline = np.zeros_like(y_true)
baseline = {
    "accuracy": accuracy_score(y_true, pred_baseline),
    "precision": precision_score(y_true, pred_baseline, zero_division=0),
    "recall": recall_score(y_true, pred_baseline, zero_division=0),
    "F1": f1_score(y_true, pred_baseline, zero_division=0),
}
assert baseline == {"accuracy": 0.94, "precision": 0.0, "recall": 0.0, "F1": 0.0}
pd.Series(baseline)

## 5. Prevalência e precision

Mantemos sensibilidade `0.90` e FPR `0.05`. Apenas a taxa-base varia. Pela regra de Bayes, a precision esperada é `TPR*pi / (TPR*pi + FPR*(1-pi))`.

In [ ]:
tpr = 0.90
fpr = 0.05
prevalencias = np.array([0.50, 0.10, 0.01, 0.001])
precision_esperada = tpr * prevalencias / (
    tpr * prevalencias + fpr * (1 - prevalencias)
)
efeito_prevalencia = pd.DataFrame(
    {"prevalencia": prevalencias, "precision_esperada": precision_esperada}
)
efeito_prevalencia

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(
    efeito_prevalencia["prevalencia"] * 100,
    efeito_prevalencia["precision_esperada"] * 100,
    marker="o",
)
ax.set(
    xlabel="Prevalência (%) — escala log",
    ylabel="Precision esperada (%)",
    title="A mesma TPR e FPR produzem precision diferente",
)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Texto alternativo do gráfico:** a precision cai de aproximadamente 94,7% com prevalência de 50% para 1,8% com prevalência de 0,1%, apesar de TPR e FPR fixas.

## 6. F-beta altera a ênfase

`beta=2` enfatiza recall; `beta=0.5`, precision. Isso não substitui uma função de custo do domínio.

In [ ]:
f_scores = pd.Series(
    {
        "F0.5": fbeta_score(y_true, y_pred, beta=0.5),
        "F1": fbeta_score(y_true, y_pred, beta=1.0),
        "F2": fbeta_score(y_true, y_pred, beta=2.0),
    }
)
assert f_scores["F0.5"] > f_scores["F1"] > f_scores["F2"]
f_scores.round(6)

Como precision (`0.8`) supera recall (`0.6667`), enfatizar precision eleva o resumo; enfatizar recall o reduz. O ranking poderia inverter em outro modelo.

## 7. Custo de uma política já fixada

O threshold não será otimizado aqui. Apenas traduzimos a matriz existente em custo, com FP a R$ 50 e FN a R$ 5.000.

In [ ]:
custo_fp = 50
custo_fn = 5_000
custo_total = custo_fp * fp + custo_fn * fn
custo_medio = custo_total / len(y_true)
assert custo_total == 100_500
print(f"Custo total: R$ {custo_total:,.2f}")
print(f"Custo médio por decisão: R$ {custo_medio:,.2f}")

## 8. Multiclasse e médias

Criamos 80 exemplos A, 15 B e 5 C. Por construção, F1 por classe será `0.9`, `0.6` e `0.2`.

In [ ]:
y_multi = np.array(["A"] * 80 + ["B"] * 15 + ["C"] * 5)
pred_multi = np.array(
    ["A"] * 72 + ["B"] * 5 + ["C"] * 3
    + ["A"] * 5 + ["B"] * 9 + ["C"] * 1
    + ["A"] * 3 + ["B"] * 1 + ["C"] * 1
)
labels = ["A", "B", "C"]
cm_multi = confusion_matrix(y_multi, pred_multi, labels=labels)
assert cm_multi.tolist() == [[72, 5, 3], [5, 9, 1], [3, 1, 1]]
pd.DataFrame(cm_multi, index=[f"real_{x}" for x in labels], columns=[f"prev_{x}" for x in labels])

In [ ]:
relatorio = classification_report(
    y_multi, pred_multi, labels=labels, output_dict=True, zero_division=0
)
resumo_multi = pd.DataFrame(relatorio).T
resumo_multi.loc[["A", "B", "C", "accuracy", "macro avg", "weighted avg"]].round(4)

A macro-F1 vale cerca de `0.5667`, enquanto a weighted-F1 vale `0.82`: o desempenho da classe A domina a média ponderada.

## 9. One-vs-rest e identidade micro

Reconstruímos C como positiva contra A+B. Em classificação multiclasse de rótulo único, micro-precision, micro-recall e micro-F1 coincidem com accuracy.

In [ ]:
y_c = (y_multi == "C").astype(int)
pred_c = (pred_multi == "C").astype(int)
tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_c, pred_c, labels=[0, 1]).ravel()

acc = accuracy_score(y_multi, pred_multi)
micro_p = precision_score(y_multi, pred_multi, average="micro")
micro_r = recall_score(y_multi, pred_multi, average="micro")
micro_f1 = f1_score(y_multi, pred_multi, average="micro")
np.testing.assert_allclose([acc, micro_p, micro_r, micro_f1], [0.82] * 4)
print({"C_TP": int(tp_c), "C_FP": int(fp_c), "C_FN": int(fn_c), "C_TN": int(tn_c)})
print(f"Accuracy = micro-P = micro-R = micro-F1 = {acc:.2f}")

## 10. Métricas por grupo

A média pode esconder diferenças. Os grupos abaixo têm tamanho e prevalência distintos; por isso reportamos também support.

In [ ]:
def grupo_binario(nome, tp, fp, fn, tn):
    return pd.DataFrame(
        {
            "grupo": nome,
            "real": np.r_[np.ones(tp), np.zeros(fp), np.ones(fn), np.zeros(tn)].astype(int),
            "prev": np.r_[np.ones(tp), np.ones(fp), np.zeros(fn), np.zeros(tn)].astype(int),
        }
    )


dados_grupo = pd.concat(
    [grupo_binario("Norte", 45, 5, 5, 45), grupo_binario("Sul", 4, 4, 6, 86)],
    ignore_index=True,
)

linhas = []
for grupo, parte in dados_grupo.groupby("grupo", sort=True):
    linhas.append(
        {
            "grupo": grupo,
            "n": len(parte),
            "positivos": int(parte["real"].sum()),
            "precision": precision_score(parte["real"], parte["prev"]),
            "recall": recall_score(parte["real"], parte["prev"]),
        }
    )
por_grupo = pd.DataFrame(linhas).set_index("grupo")
assert por_grupo.loc["Norte", "recall"] == 0.9
assert por_grupo.loc["Sul", "recall"] == 0.4
por_grupo

A diferença observada exige investigação de dados, população e incerteza; não autoriza conclusão causal automática. O suporte positivo do grupo Sul é apenas 10.

## 11. Verificações finais

In [ ]:
assert SEED == 20260908
assert len(y_true) == len(y_pred) == 1_000
assert manual["accuracy"] > manual["balanced_accuracy"]
assert manual["precision"] > manual["recall"]
assert baseline["accuracy"] == 0.94 and baseline["recall"] == 0
assert np.all(np.diff(precision_esperada) < 0)
assert custo_total == 100_500
assert np.isclose(resumo_multi.loc["macro avg", "f1-score"], 17 / 30)
assert np.isclose(resumo_multi.loc["weighted avg", "f1-score"], 0.82)
assert tp_c == 1 and fp_c == 4 and fn_c == 4
assert por_grupo.loc["Norte", "recall"] > por_grupo.loc["Sul", "recall"]
print("Todas as verificações passaram.")

## Conclusões verificadas

- Fórmulas manuais e scikit-learn coincidiram.
- Accuracy de `0.97` coexistiu com recall de `0.666667`.
- O baseline sempre negativo alcançou accuracy `0.94`, mas recall e F1 zero.
- Com TPR e FPR fixas, reduzir prevalência derrubou a precision.
- A política fixa custou R$ `100.500`, ou R$ `100,50` por decisão.
- No problema multiclasse, macro-F1 foi `0.566667`, enquanto weighted e micro-F1 foram `0.82`.
- O recall por grupo variou de `0.9` a `0.4`, com supports diferentes.

Antes de abrir o teste, registre classe positiva, prevalência, custos, tipo de média e tratamento de divisões indefinidas.